# 04 — Scenario compare (Phase 2)

What-if rebalancing on top of the Phase 1 diagnostic. Express a proposed
book via the edit helpers (`drop_portfolio`, `rebalance_into`, `merge_into`,
`set_weights`) and diff it against the current book — every Phase 1 metric
is recomputed on both sides and surfaced as deltas.

## Setup — load real book + market data

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.allocation.returns import last_business_day_on_or_before
from hailmary.allocation.scenarios import (
    Scenario, drop_portfolio, merge_into, rebalance_into, scenario_compare, set_weights,
)
from hailmary.data.providers import YahooFinanceProvider

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
START = date(2022, 1, 1)
END = last_business_day_on_or_before(date.today())

parsed = parse_statement(STATEMENT_PATH)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
holding = [p for p in portfolios if Role.HOLDING in p.roles]
tickers = sorted({
    h.metadata.ticker for p in holding for h in p.holdings
    if not h.metadata.ticker.startswith('CASH_')
})
provider = YahooFinanceProvider()
returns = provider.get_returns(tickers, START, END)
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
print(f'{len(portfolios)} portfolios loaded, {len(holding)} HOLDING-tagged, '
      f'window {START}..{END}')

In [ ]:
def headline(diff):
    d = diff.deltas
    print(f'Combined book changes ({diff.current.label}  ->  {diff.proposed.label}):')
    print(f'  Sharpe Δ:    {d.book_sharpe_delta:+.3f}')
    print(f'  Ann ret Δ:   {d.book_ann_return_delta:+.2%}')
    print(f'  Vol Δ:       {d.book_ann_vol_delta:+.2%}')
    print(f'  Max DD Δ:    {d.book_max_dd_delta:+.2%}')
    print(f'  AUM Δ:       {d.book_aum_delta:+,.0f} SGD')
    if d.redundancy_appeared:
        print(f'  Redundancy appeared:    {d.redundancy_appeared}')
    if d.redundancy_disappeared:
        print(f'  Redundancy disappeared: {d.redundancy_disappeared}')

cur = Scenario('current book', tuple(portfolios))
kw = dict(returns=returns, fx_series_usd_sgd=fx_series_usd_sgd, align_window=True)

## Scenario A — What if I dropped Crypto?

Removes the Crypto sleeve entirely. The freed capital is just *gone* from the
book (use `rebalance_into` if you want to redeploy it). Useful for seeing
how much risk Crypto is contributing.

In [ ]:
proposed = drop_portfolio(portfolios, 'Crypto')
diff_a = scenario_compare(cur, Scenario('drop Crypto', proposed), **kw)
headline(diff_a)

In [ ]:
print('Asset-class exposure shift:')
diff_a.deltas.exposure_delta['asset_class'].head(10)

## Scenario B — What if I moved Crypto into BlackRock?

`rebalance_into` moves Crypto's whole `total_value` into BlackRock
at BlackRock's current composition. Crypto disappears; BlackRock gets bigger.
Both are USD sleeves — same-currency rotations only (cross-currency rotations
raise `ScenarioEditError`; convert FX first if needed).

In [ ]:
proposed = rebalance_into(portfolios, 'Crypto', 'BlackRock')
diff_b = scenario_compare(cur, Scenario('Crypto -> BlackRock', proposed), **kw)
headline(diff_b)

## Scenario C — What if I merged Energy + Utilities + HDY into one sleeve?

Value-weighted union of the three customs into a single 'Custom Equity Sleeve'.
Same total exposure — just consolidated administratively.

In [ ]:
proposed = merge_into(
    portfolios,
    names=['Energy', 'Utilities', 'High Dividend Yield'],
    into='Custom Equity Sleeve',
)
diff_c = scenario_compare(cur, Scenario('merge customs', proposed), **kw)
headline(diff_c)

## Scenario D — What if I shifted Crypto weights 50/50 BTC/ETH?

`set_weights` replaces one sleeve's weights. Requires explicit weight for
every existing holding (pass 0.0 to zero one out).

In [ ]:
crypto = next(p for p in portfolios if p.name == 'Crypto')
print('Current Crypto holdings:')
for h in crypto.holdings:
    print(f'  {h.ticker:<10} weight={h.weight:.3f}')

In [ ]:
# Build a 50/50 BTC/ETH proposal (zero out any other holdings)
current_weights = {h.ticker: h.weight for h in crypto.holdings}
new_weights = {sid: 0.0 for sid in current_weights}
if 'FBTC' in new_weights: new_weights['FBTC'] = 0.5
if 'FETH' in new_weights: new_weights['FETH'] = 0.5
# Sanity: weights sum to 1
assert abs(sum(new_weights.values()) - 1.0) < 1e-9, new_weights

proposed = set_weights(portfolios, 'Crypto', new_weights)
diff_d = scenario_compare(cur, Scenario('Crypto 50/50 BTC/ETH', proposed), **kw)
headline(diff_d)

## Regime sanity — how does scenario B look year-by-year?

Aggregate Sharpe Δ averages over all regimes. A +0.2 headline can hide a
+0.6 great year and a -0.4 bad year. `by_period_deltas` slices the impact
by period (standard windows + each calendar year).

In [ ]:
bp = diff_b.deltas.by_period_deltas.set_index('period')
show = bp[['cur_sharpe', 'prop_sharpe', 'delta_sharpe',
           'cur_ann_return', 'prop_ann_return', 'delta_ann_return',
           'delta_max_dd']]
show.style.format({
    'cur_sharpe':'{:.2f}', 'prop_sharpe':'{:.2f}', 'delta_sharpe':'{:+.3f}',
    'cur_ann_return':'{:+.2%}', 'prop_ann_return':'{:+.2%}',
    'delta_ann_return':'{:+.2%}', 'delta_max_dd':'{:+.2%}',
}, na_rep='-')

Read the calendar-year rows: if `delta_sharpe` flips sign across years, the
aggregate is masking real regime variance — that's a signal to be careful
before acting on the headline alone. If `delta_sharpe` is consistently
positive year after year, the case is much stronger.

## Tail-risk sanity — best & worst rolling N-day stretches

Calendar years can hide the worst stretch entirely (e.g. a sharp 6-week
drawdown that lives inside a flat full year). `tail_metrics` slides a
rolling window across the whole sample and reports the best and worst
observed cumulative return for each window length — plus the date each
extreme was hit.

<strong>`delta_worst > 0`</strong> means proposed has a less-bad worst stretch
(better tail protection). <strong>`delta_best > 0`</strong> means proposed
has a better upside stretch. A scenario that improves Δ worst more than it
costs Δ best is winning on asymmetric risk.

In [ ]:
tm = diff_b.deltas.tail_metrics
tm[['cur_best','cur_best_date','cur_worst','cur_worst_date',
    'prop_best','prop_best_date','prop_worst','prop_worst_date',
    'delta_best','delta_worst']].style.format({
    'cur_best': '{:+.2%}', 'cur_worst': '{:+.2%}',
    'prop_best': '{:+.2%}', 'prop_worst': '{:+.2%}',
    'delta_best': '{:+.2%}', 'delta_worst': '{:+.2%}',
}, na_rep='-')

## Compare all four scenarios side-by-side

In [ ]:
import pandas as pd
rows = []
for label, diff in [
    ('A: drop Crypto', diff_a),
    ('B: Crypto -> SI', diff_b),
    ('C: merge customs', diff_c),
    ('D: 50/50 BTC/ETH', diff_d),
]:
    d = diff.deltas
    rows.append({
        'scenario': label,
        'Sharpe Δ': d.book_sharpe_delta,
        'AnnRet Δ': d.book_ann_return_delta,
        'Vol Δ':    d.book_ann_vol_delta,
        'MaxDD Δ':  d.book_max_dd_delta,
        'AUM Δ':    d.book_aum_delta,
        'red. appeared':    len(d.redundancy_appeared),
        'red. disappeared': len(d.redundancy_disappeared),
    })
summary = pd.DataFrame(rows).set_index('scenario')
summary.style.format({
    'Sharpe Δ': '{:+.3f}', 'AnnRet Δ': '{:+.2%}',
    'Vol Δ': '{:+.2%}', 'MaxDD Δ': '{:+.2%}', 'AUM Δ': '{:+,.0f}',
}, na_rep='-')

## Export HTML reports

`render_scenario_report(diff, path)` writes a self-contained HTML with the
headline delta strip, by-period regime table, tail-risk best/worst rolling
stretches, per-portfolio shifts, exposure deltas and the redundancy
lifecycle — same dark theme + sortable tables as the Phase 1 report.

In [ ]:
from pathlib import Path
from hailmary.allocation.scenarios import render_scenario_report

REPORTS_DIR = Path('../../reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    'A_drop_crypto': diff_a,
    'B_crypto_to_blackrock': diff_b,
    'C_merge_customs': diff_c,
    'D_crypto_50_50_btc_eth': diff_d,
}
for slug, diff in outputs.items():
    path = REPORTS_DIR / f'scenario_{slug}.html'
    render_scenario_report(diff, path, title=f'Scenario — {slug}')
    print(f'wrote {path.resolve()}')